# Day 29: Embeddings and Vector Databases

On Day 26 we put **all** the shop facts in the prompt. That is fine for 8 facts. A real shop has **hundreds**. Sending all of them with every question costs a lot of tokens.

**Better idea:** for each question, **find only the facts that help**, and send only those.

## 🎯 The example we will build

| Customer asks | Fact we find |
|---|---|
| *How much to send food to my home?* | The delivery charge is Rs 100. Delivery is free for orders of Rs 1500 or more. |

The question does **not** have the word "delivery", but we still find the right fact. We search by **meaning**, not by words.

## 📋 Agenda

```
TOP     Step 5   Search in Chroma         the database finds the closest facts
          ▲
        Step 4   Store in Chroma          a vector database
          ▲
        Step 3   Search by hand           question vs every fact
          ▲
        Step 2   Facts → vectors          embed all the shop facts
          ▲
BOTTOM  Step 1   Quick recap              embedding + similarity (Day 24)
```

### New words today

| Word | Meaning |
|---|---|
| **Embedding** | A list of numbers that describes the meaning of a text (Day 23, Day 24). |
| **Semantic search** | Search by **meaning**, not by the same words. |
| **Vector database** | A database that stores vectors and quickly finds the closest ones. |
| **Chroma** | A free vector database for Python. |
| **Collection** | A table inside Chroma, like one sheet in Excel. |
| **Distance** | How **far** two meanings are. Small distance = similar. (Distance = 1 − similarity.) |

---
# Class Problem: find the right Chai Wala fact

## Step 1: Quick recap

From Day 24:

- An **embedding model** turns a text into a long list of numbers.
- **Cosine similarity** compares 2 vectors. Close to **1** = similar meaning.

Install once: `pip install -U chromadb python-dotenv` (`-U` gets the newest version)

**Good to know:**
- A free OpenRouter account gets about **50 free requests per day**. So we embed a whole list in **one** request, and we run each cell only once.
- If Chroma prints *"Failed to send telemetry event"*, your Chroma is old. Run `pip install -U chromadb`, then **restart the kernel**. (It is not a real error. Your code still works.)
- If a cell runs twice, Chroma may say *"existing embedding ID"*. The fact is already saved. That is fine.

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv          # reads secrets from the .env file
from openai import OpenAI

MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free)

load_dotenv()                                       # finds .env and loads OPEN_ROUTER_KEY
key = os.getenv("OPEN_ROUTER_KEY") or getpass("Paste your OpenRouter key (hidden): ")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

import numpy as np

EMB_MODEL = "nvidia/nemotron-3-embed-1b:free"      # embedding model (free, Day 24)

def embed(texts):
    r = client.embeddings.create(model=EMB_MODEL, input=texts)    # one request for the whole list
    return [item.embedding for item in r.data]                    # one vector per text

def similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))     # cosine similarity (Day 24)

## Step 2: Facts → vectors

We write the Chai Wala facts as a list. **One fact per item.** Then we embed all of them in **one** request.

In [ ]:
facts = [
    "Doodh patti chai costs Rs 120 for one cup.",
    "Kashmiri chai costs Rs 200 for one cup.",
    "Samosa costs Rs 60. We have aloo samosa only.",
    "We deliver to Gulberg, DHA, Model Town and Johar Town.",
    "The delivery charge is Rs 100. Delivery is free for orders of Rs 1500 or more.",
    "We are open every day from 9 am to 11 pm.",
    "On Friday we close from 1 pm to 2 pm for Jumma prayer.",
    "You can pay with cash, card, JazzCash or Easypaisa.",
]

fact_vectors = embed(facts)
print(len(fact_vectors), "vectors")
print(len(fact_vectors[0]), "numbers in each vector")

## Step 3: Search by hand

1. Embed the **question**.
2. Compare it with **every** fact (similarity).
3. Sort: the biggest similarity is the best fact.

In [ ]:
question = "How much to send food to my home?"
q_vec = embed([question])[0]

scores = [similarity(q_vec, v) for v in fact_vectors]

for score, fact in sorted(zip(scores, facts), reverse=True)[:3]:      # top 3
    print(f"{score:.2f}  {fact}")

The question has no word "delivery" or "charge", but the delivery facts are at the top. The model **learned** that "send food to my home" means delivery.

**✏️ Quick Check:** Search for *"Can I pay with my phone?"*. Which fact comes first?

In [ ]:
# your code here

**Answer:** the payment fact (JazzCash and Easypaisa are phone apps).

In [ ]:
q_vec = embed(["Can I pay with my phone?"])[0]
scores = [similarity(q_vec, v) for v in fact_vectors]
for score, fact in sorted(zip(scores, facts), reverse=True)[:3]:
    print(f"{score:.2f}  {fact}")

**Problems with search by hand:**

1. **Slow:** 1 million facts = 1 million comparisons for every question.
2. **Lost:** close the notebook, and the vectors are gone. We must pay to embed again.
3. **Messy:** we keep the facts and the vectors in 2 lists, and must keep them in the same order.

A **vector database** solves all 3.

## Step 4: Store in Chroma

*Everyday example:* a library. Books are not on one big pile. They are on **shelves by topic**, so the librarian finds a book fast. Chroma puts our vectors "on shelves" so it finds the close ones fast.

For each fact we give Chroma 3 things:

| What | Example |
|---|---|
| `ids` | `"fact0"`: a unique name |
| `documents` | the text of the fact |
| `embeddings` | the vector of the fact |

In [ ]:
import chromadb

db = chromadb.Client()                                   # a vector database in memory
shop = db.get_or_create_collection(
    name="chai_wala",
    metadata={"hnsw:space": "cosine"},                   # measure closeness with cosine (Day 24)
)

shop.add(
    ids=[f"fact{i}" for i in range(len(facts))],         # a unique name for each fact
    documents=facts,                                     # the text
    embeddings=embed(facts),                             # the numbers
)
print(shop.count(), "facts saved")

## Step 5: Search in Chroma

We give Chroma the **question vector** and ask for the 2 closest facts (`n_results=2`).

In [ ]:
result = shop.query(
    query_embeddings=embed(["How much to send food to my home?"]),
    n_results=2,
)

for doc, dist in zip(result["documents"][0], result["distances"][0]):
    print(f"distance {dist:.2f}  {doc}")

- `result["documents"][0]`: the closest facts. (`[0]` = for our first question. We can send many questions at once.)
- `result["distances"][0]`: how far each one is. **Smaller = closer.**

**✏️ Quick Check:** Ask Chroma: *"Is the shop open on Friday afternoon?"*, with `n_results=2`. Which 2 facts come back?

In [ ]:
# your code here

**Answer:** the Friday Jumma fact and the opening hours fact.

In [ ]:
result = shop.query(query_embeddings=embed(["Is the shop open on Friday afternoon?"]), n_results=2)
for doc, dist in zip(result["documents"][0], result["distances"][0]):
    print(f"distance {dist:.2f}  {doc}")

### What about a question with no answer in the facts?

Chroma **always** returns the closest fact, even when it does not help. Compare the **distances**:

In [ ]:
for q in ["How much is Kashmiri chai?", "Do you sell pizza?", "What is the capital of Pakistan?"]:
    result = shop.query(query_embeddings=embed([q]), n_results=1)
    print(f"{result['distances'][0][0]:.2f}  {q:35} → {result['documents'][0][0]}")

- Kashmiri chai: a **small** distance. A good match.
- Capital of Pakistan: a **very big** distance. Nothing in our facts helps.
- Pizza: **in the middle**. Distance alone is not enough to decide.

On **Day 32** we use both: the distance, and a prompt rule that says "I don't know".

## ✅ Complete code (Class Problem)

All the steps in one cell. You can run this cell alone.

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv          # reads secrets from the .env file
from openai import OpenAI

MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free)

load_dotenv()                                       # finds .env and loads OPEN_ROUTER_KEY
key = os.getenv("OPEN_ROUTER_KEY") or getpass("Paste your OpenRouter key (hidden): ")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)

EMB_MODEL = "nvidia/nemotron-3-embed-1b:free"      # embedding model (free, Day 24)

def embed(texts):
    r = client.embeddings.create(model=EMB_MODEL, input=texts)    # one request for the whole list
    return [item.embedding for item in r.data]                    # one vector per text

facts = [
    "Doodh patti chai costs Rs 120 for one cup.",
    "Kashmiri chai costs Rs 200 for one cup.",
    "Samosa costs Rs 60. We have aloo samosa only.",
    "We deliver to Gulberg, DHA, Model Town and Johar Town.",
    "The delivery charge is Rs 100. Delivery is free for orders of Rs 1500 or more.",
    "We are open every day from 9 am to 11 pm.",
    "On Friday we close from 1 pm to 2 pm for Jumma prayer.",
    "You can pay with cash, card, JazzCash or Easypaisa.",
]

import chromadb

db = chromadb.Client()                                   # a vector database in memory
shop = db.get_or_create_collection(
    name="chai_wala",
    metadata={"hnsw:space": "cosine"},                   # measure closeness with cosine (Day 24)
)

shop.add(
    ids=[f"fact{i}" for i in range(len(facts))],         # a unique name for each fact
    documents=facts,                                     # the text
    embeddings=embed(facts),                             # the numbers
)
print(shop.count(), "facts saved")

def search(question, n=2):
    result = shop.query(query_embeddings=embed([question]), n_results=n)
    return result["documents"][0]

print(search("How much to send food to my home?"))

### Key takeaways

- An **embedding** turns text into numbers. Close meaning = close numbers.
- **Semantic search** finds facts by **meaning**, even with different words.
- Search by hand is slow and is lost when the notebook closes.
- A **vector database** (Chroma) stores `ids`, `documents` and `embeddings`, and finds the closest ones fast.
- Chroma **always** returns something. A **very big** distance means "not a good match".

---
# 🏠 Home Problem

Chai Wala now offers **catering** for events. Add these 3 facts to the **same** Chroma collection:

- *"Catering costs Rs 400 for one person, with chai, samosa and cake rusk."*
- *"The smallest catering order is 20 people."*
- *"Book catering at least 3 days before the event."*

1. Embed the 3 facts and `add` them to `shop`. Use new ids: `"cater0"`, `"cater1"`, `"cater2"`.
2. Check `shop.count()`. It must be 11.
3. Write a `search` function, and search these 3 questions. Print the top fact and its distance:
   - *"I have a wedding with 50 guests. Can you serve chai?"*
   - *"How early should I book for a party?"*
   - *"Can I order catering for 5 friends?"*
4. In 1 line: which question has the biggest distance? Why?

**Project piece:** write 10 facts for **your own** project, one fact per line, and store them in Chroma.